# Term Lookup Benchmarking

This notebook demonstrates how to benchmark SNOMED CT term search functionality.

## Overview

Term lookup evaluates how well a method can match clinical terms to their corresponding SNOMED CT concepts.

### Key Metrics:
- **Recall@K**: Whether the expected CUI appears in top-K results (binary: 1 or 0)
- **Precision@K**: Similarly computed
- **MRR (Mean Reciprocal Rank)**: Average of reciprocal ranks where expected CUI is found
- **Hit Rate**: Fraction of queries where expected CUI appears anywhere in results
- **Exact Match@pos_N**: Whether expected CUI is at specific rank position

### Data Format:
```python
{
    'term': str,     # Clinical term to search for
    'expected_cui': str  # Expected CUI that should be found
}
```

In [ ]:
# Import required modules
from snomed_methods.benchmarking.term import (
    evaluate_term_lookup,
    generate_term_dataset,
)

## Generate Synthetic Term Dataset

In [ ]:
# Generate term lookup dataset
dataset = generate_term_dataset(num_samples=50)

print(f"Dataset size: {len(dataset)}")
print("\nFirst 3 samples:")
for i, sample in enumerate(dataset[:3]):
    print(f"\nSample {i+1}:")
    print(f"  Term: '{sample['term']}'")
    print(f"  Expected CUI: {sample['expected_cui']}")

## Create Mock Term Lookup Function

For demonstration, we create a simple term lookup function.

In [ ]:
# Example: Mock term lookup based on hash patterns
def simple_lookup(term: str) -> list:
    """Mock lookup returning synthetic (CUI, matched_term) tuples."""
    seed_hash = hash(term)

    # Generate CUIs deterministically
    cuis = [str(abs(seed_hash + i * 1000) % 1000000000).zfill(9) for i in range(15)]

    # Return as list of (CUI, term) tuples
    return [(cui, f"Matched_{i}") for i, cui in enumerate(cuis)]


# Test the mock lookup
test_term = dataset[0]["term"]
result = simple_lookup(test_term)
print(f"Term: '{test_term}'")
print(f"Search results ({len(result)}):")
for cui, matched in result[:5]:
    print(f"  - {cui} (matched: '{matched}')")

## Evaluate Term Lookup

In [ ]:
# Evaluate term lookup
results = evaluate_term_lookup(
    lookup_func=simple_lookup,
    dataset=dataset,
    k_values=[1, 3, 5, 10],
)

print("\n=== Term Lookup Benchmarking Results ===")
for metric, value in results.items():
    if metric != "num_samples":
        print(f"{metric}: {value:.4f}")

print(f"\nTotal samples evaluated: {results['num_samples']}")

## Working with Real SNOMED Data

When actual SNOMED CT data is available, use `SnomedTermLookup`:

In [ ]:
# Example: Using real SnomedTermLookup (requires description file path)
# from snomed_methods import create_term_lookup_from_directory

# lookup = create_term_lookup_from_directory(
#     sct2_dir="/path/to/uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z"
# )

# def real_lookup(term):
#     results = lookup.find_concepts_by_term(
#         term=term,
#         top_n=20
#     )
#     return results  # List of (cui, term) tuples

# results_real = evaluate_term_lookup(real_lookup, dataset[:10])
# print(results_real)

## Load Pre-generated Datasets

In [ ]:
from snomed_methods.benchmarking.term import load_term_datasets

# Load all pre-generated datasets
datasets = load_term_datasets()

for name, data in datasets.items():
    print(f"{name}: {len(data)} samples")

## Rank Position Analysis

Analyze how often the expected CUI appears at specific positions.

In [ ]:
# Detailed rank position analysis
print("\nExact match rates at different ranks:")
for pos in [0, 1, 2, 3]:
    key = f"exact_match@pos_{pos}"
    if key in results:
        print(f"Position {pos}: {results[key]:.4f}")

## Hit Rate and MRR Analysis

In [ ]:
# Summary statistics
print("\nSummary Statistics:")
print(
    f"Hit Rate: {results.get('hit_rate', 0):.4f} ({int(results['num_samples'] * results.get('hit_rate', 0))}/{results['num_samples']} queries found)"
)
print(f"MRR: {results.get('mrr', 0):.4f}")

# Average rank where hits occur
if "mrr" in results and results["hit_rate"] > 0:
    avg_rank = 1.0 / results["mrr"]
    print(f"Average rank for hits: {avg_rank:.2f}")

## Comparison of K Values

In [ ]:
# Compare different K values for recall
k_values = [1, 2, 3, 5, 10]

results_k = evaluate_term_lookup(
    lookup_func=simple_lookup,
    dataset=dataset[:20],
    k_values=k_values,
)

print("\nRecall@K comparison:")
for k in k_values:
    r = results_k.get(f"recall@{k}", 0)
    print(f"  Recall@{k}: {r:.4f} ({int(len(dataset[:20]) * r)}/{len(dataset[:20])})")